# Why it makes things up

MichAl Academy, unit 4.7.

Run each cell with **Shift+Enter**.

A model that invents a citation is not malfunctioning. It is doing exactly what
it was trained to do, and this notebook measures the gap between what training
supplies and what a reader assumes it supplies.

The model here is **SmolLM2-135M-Instruct**: 135 million parameters, Apache 2.0,
small enough to run on a laptop CPU. It is far weaker than anything you would
deploy, which is useful, because every effect below is visible at this size and
none of it goes away at a larger one.


In [ ]:
import time
import warnings

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()
print(f"{NAME}: {sum(p.numel() for p in model.parameters()):,} parameters")


## Two different questions

Training gives the model one job: predict the next token. So it is worth asking
two separate questions about what it knows.

1. **Does it know what text is likely?** That is the thing it was trained on, and
   section 1 measures how well calibrated it is: when it says a token has
   probability 0.8, does that token turn up 80% of the time?
2. **Does it know what is true?** Nothing in training measured that. Section 2
   puts true and false statements of identical shape side by side and asks which
   one the model finds more likely.


In [ ]:
@torch.no_grad()
def token_probs(text):
    """Model probability assigned to each actual next token in `text`."""
    ids = tok(text, return_tensors="pt").input_ids
    logits = model(ids).logits[0, :-1]
    probs = logits.softmax(dim=-1)
    actual = ids[0, 1:]
    chosen = probs.argmax(dim=-1)
    return (probs.gather(1, actual[:, None])[:, 0].numpy(),
            probs.max(dim=-1).values.numpy(),
            (chosen == actual).numpy())


# Ordinary English prose the model was not trained on verbatim, used only as a
# stream of next-token questions with known answers.
SAMPLE = """The server refused the connection because the certificate had
expired three days earlier. The team had known about the renewal date for some
weeks, but nobody had been assigned to it, and the reminder went to a mailbox
that nobody reads. When the alert finally arrived it was routed to the on-call
engineer, who restarted the service twice before reading the log. The log said
what the problem was in the first line."""

p_actual, p_top, correct = token_probs(SAMPLE)
print(f"{len(p_actual)} next-token questions")
print(f"the model's top choice was right {correct.mean():.3f} of the time")


In [ ]:
# Calibration: bucket predictions by the confidence the model assigned to its own
# top choice, then measure how often that choice was actually right.
edges = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
print(f"{'confidence':>16}  {'n':>5}  {'accuracy':>9}")
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (p_top >= lo) & (p_top < hi if hi < 1.0 else p_top <= hi)
    if m.sum():
        print(f"{lo:>6.1f} to {hi:<5.1f}  {m.sum():>5}  {correct[m].mean():>9.3f}")


## 2. The same model, asked what is true

Every pair below is two statements with the same grammar and the same subject.
One is true and one is not, and the model is only asked which continuation it
finds more likely.

The pairs come in two sets, and the difference between them is the whole point.

- **Set A: the plausible wrong answer is an unrelated place.** France against
  Madrid. Nothing in ordinary text pairs France with Madrid, so the true
  statement is also the more ordinary one.
- **Set B: the plausible wrong answer is the country's most famous city**, which
  is not its capital. Australia against Sydney, Turkey against Istanbul. Here the
  false statement is made of a **more common** pairing of words than the true one.

If the model has a sense of truth, it wins both sets. If what it has is a sense
of which text is common, it wins A and loses B.


In [ ]:
# Ground truth is the internationally recognised capital in each case.
SET_A = [
    ("The capital of France is Paris.",     "The capital of France is Madrid."),
    ("The capital of Japan is Tokyo.",      "The capital of Japan is Beijing."),
    ("The capital of Egypt is Cairo.",      "The capital of Egypt is Nairobi."),
    ("The capital of Norway is Oslo.",      "The capital of Norway is Athens."),
    ("The capital of Portugal is Lisbon.",  "The capital of Portugal is Dublin."),
    ("The capital of Austria is Vienna.",   "The capital of Austria is Helsinki."),
    ("The capital of Greece is Athens.",    "The capital of Greece is Warsaw."),
    ("The capital of Poland is Warsaw.",    "The capital of Poland is Lisbon."),
]

# In every Set B pair the false city is the country's best-known city.
SET_B = [
    ("The capital of Australia is Canberra.",   "The capital of Australia is Sydney."),
    ("The capital of Turkey is Ankara.",        "The capital of Turkey is Istanbul."),
    ("The capital of Brazil is Brasilia.",      "The capital of Brazil is Rio de Janeiro."),
    ("The capital of Canada is Ottawa.",        "The capital of Canada is Toronto."),
    ("The capital of Switzerland is Bern.",     "The capital of Switzerland is Zurich."),
    ("The capital of New Zealand is Wellington.", "The capital of New Zealand is Auckland."),
    ("The capital of Nigeria is Abuja.",        "The capital of Nigeria is Lagos."),
    ("The capital of Morocco is Rabat.",        "The capital of Morocco is Casablanca."),
]


@torch.no_grad()
def logprob(text):
    """Total log probability the model assigns to this exact string."""
    ids = tok(text, return_tensors="pt").input_ids
    logits = model(ids).logits[0, :-1]
    lp = logits.log_softmax(dim=-1)
    return lp.gather(1, ids[0, 1:, None])[:, 0].sum().item()


def score(pairs):
    return [logprob(t) > logprob(f) for t, f in pairs]


a_wins, b_wins = score(SET_A), score(SET_B)
print(f"{'set':<44}{'true preferred':>15}")
print(f"{'A, wrong answer is an unrelated place':<44}"
      f"{sum(a_wins)}/{len(a_wins)} = {np.mean(a_wins):>7.3f}")
print(f"{'B, wrong answer is the famous city':<44}"
      f"{sum(b_wins)}/{len(b_wins)} = {np.mean(b_wins):>7.3f}")
print(f"{'chance':<44}{0.5:>15.3f}")
print()
for (tt, ff), w in zip(SET_B, b_wins):
    country = tt.split(" is ")[0].replace("The capital of ", "")
    print(f"  {'OK ' if w else 'NO '} {country:<14} true {logprob(tt):8.2f}   "
          f"false {logprob(ff):8.2f}")


## 3. What changes it

Nothing above changed the model. This section changes only what sits in front of
it: the same Set B pairs, scored again with the answer supplied first. This is
the whole mechanism behind retrieval, tested at the smallest possible scale.


In [ ]:
@torch.no_grad()
def logprob_given(context, text):
    full = tok(context + " " + text, return_tensors="pt").input_ids
    ctx_len = tok(context + " ", return_tensors="pt").input_ids.shape[1]
    logits = model(full).logits[0, :-1]
    lp = logits.log_softmax(dim=-1)
    per = lp.gather(1, full[0, 1:, None])[:, 0]
    return per[ctx_len - 1:].sum().item()


given = [logprob_given(f"Fact: {tt}", tt) > logprob_given(f"Fact: {tt}", ff)
         for tt, ff in SET_B]

print(f"{'Set B condition':<34}{'true preferred':>15}")
print(f"{'nothing in the context':<34}{np.mean(b_wins):>15.3f}")
print(f"{'the fact in the context':<34}{np.mean(given):>15.3f}")
print(f"{'chance':<34}{0.5:>15.3f}")


## 4. Sampling multiplies it

Section 2 scored statements the model was given. Generation is different: the
model builds the statement itself, one token at a time, and unit 4.5 showed that
raising the temperature widens the set of tokens it will accept.

The cell below asks one question forty times at three temperatures and counts
how many *distinct* answers come back. Every extra distinct answer beyond the
first is, by construction, a claim the model is willing to make and which
contradicts another claim it is also willing to make.


In [ ]:
@torch.no_grad()
def ask(question, temp, n, seed):
    msgs = [{"role": "user", "content": question}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt")
    torch.manual_seed(seed)
    out = model.generate(ids, max_new_tokens=12, do_sample=temp > 0,
                         temperature=temp if temp > 0 else None,
                         top_k=0, top_p=1.0, num_return_sequences=n,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return [tok.decode(o[ids.shape[1]:], skip_special_tokens=True).strip()
            for o in out]


# Twenty questions, not four: every rate below is a share of these twenty, and a
# rate over four questions dressed up as a rate over eighty samples would be a
# much smaller measurement than it looks.
QUESTIONS = [
    ("What is the capital of Australia?", "canberra"),
    ("What is the capital of Turkey?", "ankara"),
    ("What is the capital of Canada?", "ottawa"),
    ("What is the capital of Switzerland?", "bern"),
    ("What is the capital of Brazil?", "brasil"),
    ("What is the capital of Nigeria?", "abuja"),
    ("What is the capital of Morocco?", "rabat"),
    ("What is the capital of New Zealand?", "wellington"),
    ("What is the capital of France?", "paris"),
    ("What is the capital of Japan?", "tokyo"),
    ("What is the capital of Egypt?", "cairo"),
    ("What is the capital of Norway?", "oslo"),
    ("What is the capital of Portugal?", "lisbon"),
    ("What is the capital of Austria?", "vienna"),
    ("What is the capital of Greece?", "athens"),
    ("What is the capital of Poland?", "warsaw"),
    ("What is the capital of Kenya?", "nairobi"),
    ("What is the capital of Peru?", "lima"),
    ("What is the capital of Ireland?", "dublin"),
    ("What is the capital of Finland?", "helsinki"),
]
N_Q = len(QUESTIONS)

# "Correct" here means the expected answer appears somewhere in what came back.
# That is a generous test and worth naming as one: a reply listing three cities
# counts as correct if one of them is right.
def contains(out, answer):
    return answer in out.lower()


print(f"{N_Q} questions with one short checkable answer each")
print()
print(f"{'temperature':>12}  {'answers each':>13}  {'distinct per q':>15}  "
      f"{'share correct':>14}")

# Temperature 0 is greedy and deterministic: one answer per question, not twenty
# copies of one answer.
outs0 = {q: ask(q, 0.0, 1, seed=0)[0] for q, _ in QUESTIONS}
c0 = sum(contains(outs0[q], a) for q, a in QUESTIONS)
print(f"{0.0:>12.1f}  {1:>13}  {1.0:>15.1f}  {c0}/{N_Q} = {c0 / N_Q:>6.3f}")

SAMPLES = 20
sampled = {}
for temp in (0.7, 1.2):
    distinct, correct = [], 0
    for q, a in QUESTIONS:
        outs = ask(q, temp, SAMPLES, seed=0)
        distinct.append(len(set(o.lower() for o in outs)))
        correct += sum(contains(o, a) for o in outs)
    total = N_Q * SAMPLES
    sampled[temp] = correct / total
    print(f"{temp:>12.1f}  {SAMPLES:>13}  {np.mean(distinct):>15.1f}  "
          f"{correct}/{total} = {correct / total:>6.3f}")


Scoring a finished statement and writing one are not the same task, and the
numbers above are the second. The cell below changes nothing about the model and
only puts the answer in front of it first, which is the entire mechanism behind
retrieval.

Both rows are at temperature 0.7 so that the only difference is the context.


In [ ]:
print(f"{'condition':<30}{'share correct':>16}")
for label, prefix in (("asked cold", ""),
                      ("told the answer first", "Fact: The capital of {c} is {a}. ")):
    correct = 0
    for q, a in QUESTIONS:
        country = q.replace("What is the capital of ", "").rstrip("?")
        outs = ask(prefix.format(c=country, a=a.title()) + q, 0.7, SAMPLES, seed=0)
        correct += sum(contains(o, a) for o in outs)
    total = N_Q * SAMPLES
    print(f"{label:<30}{correct}/{total} = {correct / total:>7.3f}")

print()
print("for comparison, from the table above:")
print(f"  temperature 0, asked cold          {c0 / N_Q:.3f}")
print(f"  temperature 0.7, asked cold        {sampled[0.7]:.3f}")


## Does a shorter answer help?

The claim is that every extra token is another chance to be wrong. It is easy to
state and worth checking rather than repeating, so the cell below asks the same
twenty questions with a tight length cap and with a loose one.


In [ ]:
@torch.no_grad()
def ask_n(question, temp, n, seed, max_new):
    msgs = [{"role": "user", "content": question}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt")
    torch.manual_seed(seed)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=temp > 0,
                         temperature=temp if temp > 0 else None,
                         top_p=1.0, num_return_sequences=n,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return [tok.decode(o[ids.shape[1]:], skip_special_tokens=True).strip()
            for o in out]


print(f"{'max new tokens':>15}{'share correct':>16}")
for cap in (8, 60):
    correct = 0
    for q, a in QUESTIONS:
        correct += sum(contains(o, a) for o in ask_n(q, 0.7, SAMPLES, 0, cap))
    print(f"{cap:>15}{correct}/{N_Q * SAMPLES} = {correct / (N_Q * SAMPLES):>7.3f}")


## What this unit measured

- **The model is reasonably calibrated about text and close to chance about
  truth.** Section 1 and section 2 are the same model, the same weights, asked
  two different questions. Only one of them was ever trained for.
- **Putting the fact in the context moves the second number and not the first.**
  Section 3 changes no weights at all.
- **Sampling widens the set of statements the model will make.** Section 4 counts
  them, and every distinct answer past the first is one the model also
  contradicts.
